# Comparando celdas analizadas vs resto de celdas

Este notebook compara las celdas incluidas en el análisis (unidades tratadas y de control emparejadas) con el resto de las celdas de la ciudad que fueron excluidas. El objetivo es verificar si existe alguna diferencia sistemática entre las celdas analizadas y las excluidas, medida en términos de tasas de accidentes antes y después de la implementación de las cámaras.

**Inputs necesarios:**
- `matched-grids.parquet`: Grids que fueron emparejados en el análisis
- `outcome-variables.parquet`: Variables de resultado (accidentes) por grid y tiempo

**Outputs generados:**
- `graphs/cambios-relativos-tasa-accidentes-antes-implementacion.png`: Gráfico comparando cambios relativos en tasas de accidentes entre celdas incluidas y excluidas
- `graphs/diferencia-porcentual-tasa-accidentes-antes-implementacion.png`: Gráfico de diferencia porcentual entre celdas incluidas y excluidas

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import geopandas as gpd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore")
PATH_DATA = '../../data/'
INICIO_OPERACIONES = pd.to_datetime("2019-04-22")

In [2]:
matched_grids = pd.read_parquet(os.path.join(PATH_DATA, "matched-grids.parquet"))
outcome = pd.read_parquet(os.path.join(PATH_DATA, "outcome-variables.parquet"))

In [3]:
# ¿Cuantas celda tengo?
outcome.grid_id.unique().size

3379

In [4]:
# Ahora quiero estudiar cómo cambió a partir de que se implementaron las cámaras de velocidad
# Voy a calcular el promedio antes de la implementación y tomar como 1, de ahí calculo cómo se ve con esa referencia 
# a partir de que se pusieron las cámaras en el tratamiento y el control
cambios = (
    outcome
    .assign(
        grid_type=lambda x: x.grid_id.isin(matched_grids.grid_id.unique()).astype(int).map({0:"excluded", 1:"included"})
    )
    .pivot_table(
        index=["timestamp", "volumen_mensual"],
        columns="grid_type",
        values="total",
        aggfunc="mean"
    )
    .reset_index()
    # Convertimos a tasas
    .assign(
        excluded=lambda x: x.excluded / x.volumen_mensual,
        included=lambda x: x.included / x.volumen_mensual
    )
    # Y calculamos la diferencia proporcional con respecto al promedio
    .assign(
        mean_excluded_before_treatment=lambda df: df.loc[df.timestamp < INICIO_OPERACIONES].excluded.mean(),
        mean_included_before_treatment=lambda df: df.loc[df.timestamp < INICIO_OPERACIONES].included.mean(),
        excluded=lambda x: x.excluded / x.mean_excluded_before_treatment,
        included=lambda x: x.included / x.mean_included_before_treatment,
        change=lambda x: x.included / x.excluded - 1 
    )
    # Nos quedamos solo con los datos posteriores al tratamiento
    .pipe(lambda df: df.loc[df.timestamp >= INICIO_OPERACIONES])
    .set_index("timestamp")
    [["excluded", "included", "change"]]
)

In [5]:
def clean_ax(ax):
    ax.spines.top.set_visible(False)
    ax.spines.right.set_visible(False)

    axis_color = "#979dac"
    ax.xaxis.label.set_color(axis_color)
    ax.yaxis.label.set_color(axis_color)
    ax.spines.bottom.set_color(axis_color)
    ax.spines.left.set_color(axis_color)
    ax.tick_params(axis='both', colors=axis_color)
    ax.grid(axis='y', alpha=.2, linestyle=':')
    
    ttl = ax.title
    ttl.set_position([.5, 2])

In [47]:
fig, ax = plt.subplots(figsize=(10,4))
clean_ax(ax)

fig.suptitle("Tasa de accidentes mensual", ha="left", x=0.064, y=.935, color="gray")
ax.set_title("Cambio relativo al promedio antes de implementación", color='darkgray', loc='left', fontsize=9)
ax.set_ylabel("Porcentaje relativo")

ax.plot(
    cambios.index,
    cambios.excluded,
    color='#284b63',
    label='Excluídas',
    linewidth=.8,
    linestyle='--'
)

ax.plot(
    cambios.index,
    cambios.included,
    color='#284b63',
    linewidth=1.5,
    alpha=.8,
    label='Incluídas'
)

ax.axhline(
    y=1,
    zorder=-10, color="#495057", 
    linestyle=":", 
    linewidth=1.3, alpha=.6
)

min_y = cambios.excluded.min()
max_y = cambios.included.max()
ax.legend(ncols=2, frameon=False, loc='lower left')

fig.tight_layout()
fig.savefig(os.path.join(PATH_DATA, "graphs", "cambios-relativos-tasa-accidentes-antes-implementacion.png"), dpi=300, transparent=True)
plt.close()

In [42]:
fig, ax = plt.subplots(figsize=(10,4))
clean_ax(ax)

fig.suptitle("Diferencia porcentual", ha="left", x=0.087, y=.935, color="gray")
ax.set_title("Unidades incluídas vs excluídas", color='darkgray', loc='left', fontsize=9)
ax.set_ylabel("Diferncia (%)")

delta = (cambios.index[1] - cambios.index[0]).days
width = delta * 0.7

ax.bar(
    cambios.index,
    cambios.change,
    width=width,
    align='center',
    color='#284b63',
    alpha=.8
)

fig.tight_layout()
fig.savefig(os.path.join(PATH_DATA, "graphs", "diferencia-porcentual-tasa-accidentes-antes-implementacion.png"), dpi=300, transparent=True)
plt.close()

In [52]:
cambios.assign(negativo=lambda x: x.change.apply(lambda s: s<0).astype(int)).negativo.sum()

28

In [54]:
cambios.shape

(36, 3)